# SAM 3 polygon danger zone — lower-mask support candidates

This experiment is based on `tracking_clean_with_polygon_dangerzone.ipynb`. Instead of one bottom-center point, it extracts one or two geometric support candidates from the lower part of each person mask and tests every candidate independently. These candidates are image-left/image-right supports, not anatomical foot labels.

Rules: (1) use two separate lower-mask components when available, (2) split one wide joined component around its horizontal median, (3) keep one narrow component as one candidate, and (4) report an uncertain state when no candidate is found.


In [ ]:
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import sam3
import torch
from IPython.display import Video, display
from sam3.model_builder import build_sam3_video_predictor
from sam3.visualization_utils import load_frame

sam3_root = Path(sam3.__file__).resolve().parent.parent
utils_path = sam3_root / "examples" / "polygon_dangerzone_methods.py"
spec = importlib.util.spec_from_file_location("polygon_dangerzone_methods", utils_path)
methods = importlib.util.module_from_spec(spec)
spec.loader.exec_module(methods)

gpus_to_use = range(torch.cuda.device_count())
predictor = build_sam3_video_predictor(gpus_to_use=gpus_to_use)


## Load the video and draw or reuse the polygon

Set `redraw_zone=True` to redraw the polygon. The resulting JSON is shared with the contact-area notebook so both methods use exactly the same danger zone.


In [ ]:
video_path = sam3_root / "assets" / "videos" / "factory_2.mp4"
zone_config_path = sam3_root / "examples" / "danger_zone_factory_2.json"
output_video_path = (
    sam3_root / "examples" / "sam3_polygon_support_points.mp4"
)
zone_frame_idx = 0
redraw_zone = False

video_frames = methods.load_video_frames(video_path)
if not video_frames:
    raise RuntimeError(f"No frames were loaded from: {video_path}")

if redraw_zone or not zone_config_path.exists():
    reference_frame = load_frame(video_frames[zone_frame_idx])
    normalized_polygon = methods.select_polygon_on_frame(reference_frame)
    methods.save_zone_config(zone_config_path, normalized_polygon, video_path)
else:
    normalized_polygon = methods.load_zone_config(zone_config_path)
    print(f"Loaded danger-zone config: {zone_config_path}")

preview = methods.preview_polygon(
    load_frame(video_frames[zone_frame_idx]), normalized_polygon
)
plt.figure(figsize=(12, 7))
plt.imshow(preview)
plt.title("Shared polygon danger zone")
plt.axis("off")
plt.show()


## Run SAM 3 person tracking


In [ ]:
response = predictor.handle_request(
    request={"type": "start_session", "resource_path": str(video_path)}
)
session_id = response["session_id"]

_ = predictor.handle_request(
    request={"type": "reset_session", "session_id": session_id}
)
response = predictor.handle_request(
    request={
        "type": "add_prompt",
        "session_id": session_id,
        "frame_index": 0,
        "text": "person",
    }
)
print("Person prompt added.")


## Export using independent support candidates

Tune `lower_ratio` when too much or too little of the lower body is included. `split_width_ratio` is implemented in the shared extractor and controls when a joined component is treated as two supports. Green/red circles labeled C1 and C2 are the extracted candidates.


In [ ]:
outputs_per_frame = methods.propagate_in_video(predictor, session_id)

intrusion_events = methods.export_with_support_candidates(
    video_frames=video_frames,
    outputs_per_frame=outputs_per_frame,
    normalized_polygon=normalized_polygon,
    output_path=output_video_path,
    fps=methods.get_video_fps(video_path),
    lower_ratio=0.30,
    bottom_band=5,
    min_component_pixels=12,
    split_width_ratio=0.28,
    zone_margin=8.0,
    confirm_frames=3,
    exit_confirm_frames=3,
)

print(intrusion_events)
display(Video(str(output_video_path), embed=False))


## Release resources


In [ ]:
_ = predictor.handle_request(
    request={"type": "close_session", "session_id": session_id}
)
predictor.shutdown()
